# Fase 5 · publicação verificada
Formulário e agente usam o verificador. O chat usa o mesmo fluxo final. Agent e LangSmith remotos esperam credenciais.

In [1]:
from pathlib import Path
import os
if Path.cwd().name == 'notebooks': os.chdir('..')

In [2]:
from pedro.catalog import criar_cenario
from pedro.contracts import Solicitacao
from pedro.verification import resposta_local, verificar
from pedro.agent import renderizar, criar_fluxo
from pedro.memory import nova_sessao, Historico
cenario = criar_cenario(Path('storage/notebooks/catalogos'))
solicitacao = Solicitacao(produto_id='PEC-01-0001', quantidade=5,
    orcamento_centavos=1_000_000, prazo_max_dias=7)
resposta = resposta_local(cenario, solicitacao)
print('erros', verificar(resposta, cenario))
if not verificar(resposta, cenario): print(renderizar(resposta))
memoria = Path('storage/notebooks/final.sqlite3')
sessao = nova_sessao(memoria, 'aluna', cenario.dataset_id)
fluxo = criar_fluxo(cenario, memoria)
print(fluxo.invoke({'input':'lista peças'}, config={'configurable':{
    'user_id':'aluna','session_id':sessao}}).content[:300])
print([m.type for m in Historico(memoria, 'aluna', sessao).todos()])

erros []
CentroPeças: R$ 7.279,83 para 5 unidades; prazo de 2 dias.
- CentroPeças (nacional): R$ 7.279,83; atende [PEC-01-0001-BR-06]
- Litoral Parts (nacional): R$ 1.930,40; descartada por estoque, prazo [PEC-01-0001-BR-05]
- Atlantic Repair Parts (internacional): R$ 7.344,92; descartada por prazo [PEC-01-0001-INT-06]
Histórico de preços (informativo; não muda a recomendação):
- CentroPeças: hoje R$ 1.438,92 por unidade; mês anterior R$ 1.189,87 (+21 %); média de 5 meses R$ 1.372,51 (+5 %).
Fontes: PEC-01-0001-BR-06, PEC-01-0001-BR-05, PEC-01-0001-INT-06
Peças do cenário (4320; página 1):
- PEC-01-0001 · GELA-COMP-0001 · Compressor para Geladeiras GELA-COMP-0001
- PEC-01-0002 · GELA-TERM-0002 · Termostato para Geladeiras GELA-TERM-0002
- PEC-01-0003 · GELA-EVAP-0003 · Evaporador para Geladeiras GELA-EVAP-0003
- PEC-01-0004 · GELA-COND-0004 · Condens
['human', 'ai']


/tmp/ipykernel_455704/3946499047.py:14: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  fluxo = criar_fluxo(cenario, memoria)


## Comparação de fornecedores: pedido mínimo, origem e preferência
Execução real local, sem modelo. É o núcleo do assistente: para cada oferta conferem-se estoque, **pedido mínimo**, prazo e orçamento, e mostra-se se o fornecedor é nacional ou internacional. Uma preferência salva do usuário ordena as ofertas válidas, nunca faz vencer uma que não atende, e a resposta diz quanto custa respeitá-la.

In [3]:
from pedro.business import origem_preferida
casos = [('2 unidades: o fornecedor mais barato com estoque exige um pedido mínimo de 5',
          Solicitacao(produto_id='PEC-01-0001', quantidade=2)),
         ('5 unidades: o pedido mínimo já é alcançado',
          Solicitacao(produto_id='PEC-01-0001', quantidade=5)),
         ('sem preferência: vence a mais barata',
          Solicitacao(produto_id='PEC-11-0001', quantidade=2)),
         ('com a preferência salva «fornecedor = nacionais»',
          Solicitacao(produto_id='PEC-11-0001', quantidade=2,
                      origem_preferida=origem_preferida({'fornecedor': 'nacionais'})))]
for titulo, pedido in casos:
    comparacao = resposta_local(cenario, pedido)
    print('###', titulo, '| erros de verificação:', verificar(comparacao, cenario))
    print(renderizar(comparacao))
    print()

### 2 unidades: o fornecedor mais barato com estoque exige um pedido mínimo de 5 | erros de verificação: []
Atlantic Repair Parts: R$ 3.200,27 para 2 unidades; prazo de 35 dias.
- Atlantic Repair Parts (internacional): R$ 3.200,27; atende [PEC-01-0001-INT-06]
- Litoral Parts (nacional): R$ 819,80; descartada por estoque, pedido mínimo (5 unidades) [PEC-01-0001-BR-05]
- CentroPeças (nacional): R$ 2.963,07; descartada por pedido mínimo (5 unidades) [PEC-01-0001-BR-06]
Histórico de preços (informativo; não muda a recomendação):
- Atlantic Repair Parts: hoje R$ 1.381,55 por unidade; mês anterior R$ 1.466,10 (-6 %); média de 5 meses R$ 1.454,84 (-5 %).
Fontes: PEC-01-0001-INT-06, PEC-01-0001-BR-05, PEC-01-0001-BR-06

### 5 unidades: o pedido mínimo já é alcançado | erros de verificação: []
CentroPeças: R$ 7.279,83 para 5 unidades; prazo de 2 dias.
- CentroPeças (nacional): R$ 7.279,83; atende [PEC-01-0001-BR-06]
- Atlantic Repair Parts (internacional): R$ 7.344,92; atende [PEC-01-0001-INT-0

## Aviso de preço: função hardcoded, conselho do modelo e verificação
O histórico de preços não muda o vencedor, mas gera um aviso quando o preço de hoje se afasta da sua média. O sinal, os fatos e o aviso de modelo fixo são funções *hardcoded* e são executados aqui de verdade. **Duplo de teste:** o conselho é devolvido por um modelo roteirizado e o NLI é um classificador roteirizado, para ver os três desfechos: conselho respaldado, conselho com um número inventado e conselho sem respaldo NLI. A medição com o modelo e o NLI reais está em `docs/rastros/precos_conselho_llm.json`.

In [4]:
import json
from langchain_core.runnables import RunnableLambda
from pedro.precos import aconselhar, aviso_de_preco, criar_conselheiro, fatos_de_preco, sinal_preco
cara = resposta_local(cenario, Solicitacao(produto_id='PEC-02-0189', quantidade=5))
for alternativa in cara.alternativas:
    if alternativa.atende:
        print('sinal:', sinal_preco(alternativa.oferta).model_dump())
print('fatos:')
for fato in fatos_de_preco(cara):
    print('  -', fato)
print('aviso de modelo fixo:', aviso_de_preco(cara))
class NLIRoteirizado:
    def __init__(self, rotulo): self.rotulo = rotulo
    def classificar(self, premissa, hipotese): return self.rotulo
bom = ('O preço de CentroPeças está 11 % acima da sua média de 5 meses. Atlantic Repair Parts '
       'custa 1 % a mais que a recomendada e está 10 % abaixo da sua média.')
casos = [('conselho respaldado', bom, 'entailment'),
         ('número inventado', bom.replace('11 %', '40 %'), 'entailment'),
         ('sem respaldo NLI', bom + ' É uma grande oportunidade.', 'neutral')]
for titulo, texto, rotulo in casos:
    conselheiro = criar_conselheiro(RunnableLambda(lambda _, t=texto: json.dumps({'conselho': t})))
    resultado = aconselhar(cara, conselheiro, NLIRoteirizado(rotulo))
    print('###', titulo, '->', resultado['fonte'], '| erros:', resultado['erros'])
    print('    publica-se:', resultado['texto'])
print(renderizar(cara).splitlines()[-2])

sinal: {'fornecedor': 'CentroPeças', 'nivel': 'acima', 'frente_a_media': 11, 'frente_a_mes_anterior': 6, 'no_minimo': False, 'no_maximo': False}
sinal: {'fornecedor': 'Atlantic Repair Parts', 'nivel': 'abaixo', 'frente_a_media': -10, 'frente_a_mes_anterior': -3, 'no_minimo': True, 'no_maximo': False}
fatos:
  - A oferta recomendada é a de CentroPeças, com um total de R$ 2.182,51 e entrega em 2 dias.
  - O preço de CentroPeças está 11 % acima da sua média de 5 meses.
  - O preço de Atlantic Repair Parts está 10 % abaixo da sua média de 5 meses e é o mais baixo do período.
  - A oferta de Atlantic Repair Parts também atende: custa R$ 12,89 a mais que a recomendada (1 % a mais) e entrega em 23 dias.
aviso de modelo fixo: O preço de CentroPeças está 11 % acima da sua média de 5 meses. Atlantic Repair Parts custa R$ 12,89 a mais (1 %) e o seu preço está 10 % abaixo da sua média.
### conselho respaldado -> conselho verificado | erros: []
    publica-se: O preço de CentroPeças está 11 % acima

A avaliação remota é executada com evals/avaliar_fluxo.py. Não se reportam métricas sem essa execução.